# Chapter 5: Training Mathematics for LLMs

This chapter covers the mathematical foundations of training large language models: loss functions, regularization, optimization tricks, and scaling laws.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")


## 5.1 Training Objective

The standard language modelling objective is the negative log-likelihood averaged over the corpus $\mathcal{D}$:

$$\mathcal{L}(\theta) = -\frac{1}{|\mathcal{D}|} \sum_{x \in \mathcal{D}} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_{<t})$$

**Perplexity** is the exponentiated cross-entropy loss:

$$\text{PPL} = \exp(\mathcal{L})$$

Lower perplexity means the model assigns higher probability to the true next token. A perplexity of $k$ means the model is, on average, as confused as if it had to choose uniformly among $k$ options.


In [ ]:
# Minimal training loop: tiny transformer-like model
class TinyLM(nn.Module):
    def __init__(self, vocab_size=64, d_model=32, seq_len=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        T = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.embed(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=mask, is_causal=True)
        h = self.ln1(h + attn_out)
        h = self.ln2(h + self.ff(h))
        return self.head(h)

VOCAB, SEQ, BATCH = 64, 16, 8
model = TinyLM(vocab_size=VOCAB, d_model=32, seq_len=SEQ)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Synthetic data: random token sequences
data = torch.randint(0, VOCAB, (BATCH * 10, SEQ + 1))

total_loss = 0.0
for i in range(0, len(data), BATCH):
    batch = data[i:i+BATCH]
    x, y = batch[:, :-1], batch[:, 1:]          # input / target shifted by 1
    logits = model(x)                            # (B, T, V)
    loss = criterion(logits.reshape(-1, VOCAB), y.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

avg_loss = total_loss / (len(data) // BATCH)
ppl = math.exp(avg_loss)
print(f"Average NLL loss : {avg_loss:.4f}")
print(f"Perplexity (PPL) : {ppl:.2f}")


## 5.2 Label Smoothing

Hard cross-entropy targets drive the model to predict probability 1 for the correct token, leading to overconfident logits. **Label smoothing** mixes the one-hot target with the uniform distribution:

$$\mathcal{L}_{LS} = (1 - \varepsilon)\,\mathcal{L}_{CE} + \varepsilon\, H_{\text{uniform}}$$

where $\varepsilon \in [0,1]$ is the smoothing factor and $H_{\text{uniform}} = -\frac{1}{V}\sum_v \log P_\theta(v)$.

Benefits:
- Prevents the model from becoming overconfident.
- Acts as a soft regularizer on the output distribution.
- Empirically improves calibration and generalisation (Szegedy et al., 2016).


In [ ]:
torch.manual_seed(42)

VOCAB_LS = 32

def make_model():
    return nn.Sequential(
        nn.Linear(16, 64), nn.ReLU(),
        nn.Linear(64, VOCAB_LS)
    )

model_std = make_model()
model_ls  = make_model()
# Copy identical weights so comparison is fair
model_ls.load_state_dict(model_std.state_dict())

criterion_std = nn.CrossEntropyLoss(label_smoothing=0.0)
criterion_ls  = nn.CrossEntropyLoss(label_smoothing=0.1)

x      = torch.randn(16, 16)
target = torch.randint(0, VOCAB_LS, (16,))

# Standard loss backward
logits_std = model_std(x)
loss_std   = criterion_std(logits_std, target)
loss_std.backward()
gnorm_std = sum(p.grad.norm().item()**2 for p in model_std.parameters() if p.grad is not None) ** 0.5

# Label-smoothed loss backward
logits_ls = model_ls(x)
loss_ls   = criterion_ls(logits_ls, target)
loss_ls.backward()
gnorm_ls = sum(p.grad.norm().item()**2 for p in model_ls.parameters() if p.grad is not None) ** 0.5

print(f"Standard CE   — loss: {loss_std.item():.4f}  |  gradient norm: {gnorm_std:.4f}")
print(f"Label-Smooth  — loss: {loss_ls.item():.4f}  |  gradient norm: {gnorm_ls:.4f}")
print(f"Gradient norm ratio (LS/Std): {gnorm_ls/gnorm_std:.4f}")


## 5.3 Weight Initialization

Proper weight initialization keeps activations and gradients at stable variance across layers, preventing vanishing or exploding signals at the start of training.

**Xavier / Glorot** (sigmoid / tanh activations):
$$\sigma = \sqrt{\frac{2}{n_{\text{in}} + n_{\text{out}}}}$$

**He / Kaiming** (ReLU activations):
$$\sigma = \sqrt{\frac{2}{n_{\text{in}}}}$$

**GPT-2 residual scaling**: projection layers inside residual branches are scaled down by $\frac{1}{\sqrt{N_L}}$ where $N_L$ is the number of residual layers, preventing the variance from growing with depth.


In [ ]:
torch.manual_seed(42)

def build_net(init_fn=None):
    layers = []
    dims = [64, 64, 64, 64, 64]   # 4 linear layers
    for i in range(len(dims) - 1):
        lin = nn.Linear(dims[i], dims[i+1], bias=False)
        if init_fn is not None:
            init_fn(lin.weight)
        layers.append(lin)
        layers.append(nn.ReLU())
    return nn.Sequential(*layers)

def get_variances(net, x):
    variances = []
    h = x
    for layer in net:
        h = layer(h)
        if isinstance(layer, nn.ReLU):
            variances.append(h.var().item())
    return variances

x_in = torch.randn(256, 64)

net_default = build_net()
net_xavier  = build_net(nn.init.xavier_uniform_)
net_he      = build_net(lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'))

var_default = get_variances(net_default, x_in)
var_xavier  = get_variances(net_xavier,  x_in)
var_he      = get_variances(net_he,      x_in)

print("Activation variance after each ReLU layer")
print(f"{'Layer':<8} {'Default':>12} {'Xavier':>12} {'He/Kaiming':>12}")
print("-" * 48)
for i, (d, x, h) in enumerate(zip(var_default, var_xavier, var_he), 1):
    print(f"{i:<8} {d:>12.4f} {x:>12.4f} {h:>12.4f}")


## 5.4 Regularization — Weight Decay & Dropout

**L2 Regularization** adds a penalty $\frac{\lambda}{2}\|\mathbf{w}\|^2$ to the loss. Its gradient contribution is $\lambda\mathbf{w}$, shrinking weights toward zero each update.

**AdamW** (Loshchilov & Hutter, 2019) decouples weight decay from the adaptive learning rate scaling, applying the decay directly to the parameters:
$$\mathbf{w}_{t+1} = (1 - \eta\lambda)\,\mathbf{w}_t - \eta\,\hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$$

Bias terms and LayerNorm parameters should **not** receive weight decay, because they do not contribute to the L2 norm in a meaningful way.

**Dropout** randomly zeroes activations at rate $p$ during training, then rescales during eval (inverted dropout):
$$\tilde{h}_i = h_i \cdot \frac{\text{Bernoulli}(1 - p)}{1 - p}$$


In [ ]:
torch.manual_seed(42)

class RegNet(nn.Module):
    def __init__(self, dropout_p=0.1):
        super().__init__()
        self.ln    = nn.LayerNorm(32)
        self.fc1   = nn.Linear(32, 64)
        self.drop  = nn.Dropout(dropout_p)
        self.fc2   = nn.Linear(64, 32)
        self.bias_param = nn.Parameter(torch.zeros(32))  # explicit bias

    def forward(self, x):
        x = self.ln(x)
        x = F.gelu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        return x + self.bias_param

net = RegNet(dropout_p=0.1)

# Param groups: no decay for bias / LayerNorm
no_decay_names = {'bias', 'weight', 'bias_param'}  # LN weight is also excluded
decay_params    = [p for n, p in net.named_parameters() if 'ln' not in n and 'bias' not in n]
no_decay_params = [p for n, p in net.named_parameters() if 'ln' in n or 'bias' in n]

optimizer = optim.AdamW([
    {'params': decay_params,    'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=1e-3)

print("Parameter groups:")
print(f"  Decay group    : {sum(p.numel() for p in decay_params):>6} params, wd=0.1")
print(f"  No-decay group : {sum(p.numel() for p in no_decay_params):>6} params, wd=0.0")

x = torch.randn(8, 32)

# Dropout: train mode zeros activations
net.train()
out_train = net(x)
sparsity_train = (out_train == 0).float().mean().item()

# Dropout: eval mode passes all activations
net.eval()
with torch.no_grad():
    out_eval = net(x)

print(f"\nDropout train output zero fraction : {sparsity_train:.3f}")
print(f"Dropout eval  output shape         : {out_eval.shape}")


## 5.5 Gradient Clipping

In deep networks gradients can explode, destabilising training. Gradient clipping rescales the entire gradient vector when its norm exceeds a threshold $\tau$:

$$\mathbf{g} \leftarrow \mathbf{g} \cdot \frac{\tau}{\max(\tau,\, \|\mathbf{g}\|_2)}$$

This preserves the direction of the gradient while bounding its magnitude. Values of $\tau \in [0.5, 5]$ are typical for LLM training (GPT-3 uses $\tau = 1.0$).


In [ ]:
torch.manual_seed(42)

class DeepNet(nn.Module):
    def __init__(self, depth=12):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(32, 32) for _ in range(depth)])

    def forward(self, x):
        for layer in self.layers:
            x = torch.tanh(layer(x))   # tanh saturates, can cause issues
        return x.sum()

def get_grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5

net_unclipped = DeepNet(depth=12)
net_clipped   = DeepNet(depth=12)
net_clipped.load_state_dict(net_unclipped.state_dict())

# Simulate 5 steps
print(f"{'Step':<6} {'Unclipped norm':>16} {'Clipped norm':>14}")
print("-" * 40)
for step in range(5):
    x = torch.randn(4, 32) * 3.0   # amplified input to provoke large gradients

    # Unclipped
    opt_u = optim.SGD(net_unclipped.parameters(), lr=1e-2)
    opt_u.zero_grad()
    net_unclipped(x).backward()
    norm_u = get_grad_norm(net_unclipped)
    opt_u.step()

    # Clipped
    opt_c = optim.SGD(net_clipped.parameters(), lr=1e-2)
    opt_c.zero_grad()
    net_clipped(x).backward()
    nn.utils.clip_grad_norm_(net_clipped.parameters(), max_norm=1.0)
    norm_c = get_grad_norm(net_clipped)
    opt_c.step()

    print(f"{step+1:<6} {norm_u:>16.4f} {norm_c:>14.4f}")


## 5.6 Mixed Precision Training

Floating-point formats trade range against precision:

| Format | Sign | Exponent bits | Mantissa bits | Dynamic range |
|--------|------|--------------|---------------|---------------|
| FP32   | 1    | 8            | 23            | ~1.2e-38 – 3.4e38 |
| FP16   | 1    | 5            | 10            | ~6e-5  – 65504    |
| BF16   | 1    | 8            | 7             | same as FP32  |

**FP16** has limited range and requires **loss scaling** to prevent underflow of gradients. A `GradScaler` multiplies the loss by a large factor $S$ before the backward pass and divides gradients back by $S$ before the optimiser step.

**BF16** shares FP32's exponent width, so gradients rarely underflow — no loss scaling is needed. It is the preferred format for modern LLM training (A100/H100 hardware).


In [ ]:
import time
torch.manual_seed(42)

class MediumNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(256, 512), nn.GELU(),
            nn.Linear(512, 512), nn.GELU(),
            nn.Linear(512, 256),
        )
    def forward(self, x):
        return self.net(x)

net_fp32 = MediumNet()
net_bf16 = MediumNet()
net_bf16.load_state_dict(net_fp32.state_dict())

x = torch.randn(128, 256)

# FP32 forward
t0 = time.perf_counter()
for _ in range(20):
    out_fp32 = net_fp32(x)
t_fp32 = (time.perf_counter() - t0) / 20 * 1000

# BF16 autocast forward
t0 = time.perf_counter()
for _ in range(20):
    with torch.autocast('cpu', dtype=torch.bfloat16):
        out_bf16 = net_bf16(x)
t_bf16 = (time.perf_counter() - t0) / 20 * 1000

print(f"FP32  output dtype : {out_fp32.dtype}  | avg time: {t_fp32:.3f} ms")
print(f"BF16  output dtype : {out_bf16.dtype} | avg time: {t_bf16:.3f} ms")

# GradScaler pattern for FP16 (CPU demonstration — no actual FP16 CUDA here)
print("\n--- GradScaler pattern (FP16) ---")
scaler = torch.cuda.amp.GradScaler(enabled=False)  # enabled=False for CPU demo
opt = optim.Adam(net_fp32.parameters(), lr=1e-4)
loss = net_fp32(x).sum()
scaler.scale(loss).backward()
scaler.step(opt)
scaler.update()
print(f"GradScaler scale factor (demo): {scaler.get_scale()}")
print("Pattern: scaler.scale(loss).backward() -> scaler.step(opt) -> scaler.update()")


## 5.7 Gradient Checkpointing

During a standard forward pass all intermediate activations are stored for the backward pass. Memory scales as $O(L \cdot T \cdot d)$ where $L$ is the number of layers, $T$ the sequence length, and $d$ the model width.

**Gradient checkpointing** (Chen et al., 2016) trades compute for memory. Activations are discarded after the forward pass and recomputed on demand during backprop. Optimal recomputation at every $\sqrt{L}$ layers gives:

$$\text{Memory} = O\!\left(\sqrt{L} \cdot T \cdot d\right)$$

at the cost of one extra forward pass, roughly doubling compute while cutting memory by $\sqrt{L}$ times.


In [ ]:
from torch.utils.checkpoint import checkpoint
torch.manual_seed(42)

class CheckpointBlock(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d, d * 2), nn.GELU(), nn.Linear(d * 2, d)
        )

    def forward(self, x):
        return x + self.fc(x)

class DeepSeq(nn.Module):
    def __init__(self, n_blocks=8, d=128, use_checkpoint=False):
        super().__init__()
        self.blocks = nn.ModuleList([CheckpointBlock(d) for _ in range(n_blocks)])
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        for block in self.blocks:
            if self.use_checkpoint:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return x

x = torch.randn(16, 128, requires_grad=False)

net_std = DeepSeq(n_blocks=8, use_checkpoint=False)
net_ckpt = DeepSeq(n_blocks=8, use_checkpoint=True)
net_ckpt.load_state_dict(net_std.state_dict())

# Forward + backward without checkpointing
x1 = x.clone().requires_grad_(True)
out_std = net_std(x1)
loss_std = out_std.sum()
loss_std.backward()

# Forward + backward with checkpointing
x2 = x.clone().requires_grad_(True)
out_ckpt = net_ckpt(x2)
loss_ckpt = out_ckpt.sum()
loss_ckpt.backward()

# Verify gradients are equal
for (n1, p1), (n2, p2) in zip(net_std.named_parameters(), net_ckpt.named_parameters()):
    if p1.grad is not None:
        assert torch.allclose(p1.grad, p2.grad, atol=1e-5), f"Grad mismatch at {n1}"

print("Standard forward  output norm    :", out_std.norm().item())
print("Checkpointed output norm         :", out_ckpt.norm().item())
print("Gradients match between modes    : True")
print(f"Memory savings (theoretical)     : O(sqrt({8})) = O({8**0.5:.1f}) layers vs O({8}) layers")


## 5.8 LoRA Fine-tuning

**Low-Rank Adaptation** (Hu et al., 2021) freezes the pre-trained weight matrix $\mathbf{W}_0 \in \mathbb{R}^{d_{out} \times d_{in}}$ and injects a trainable low-rank update:

$$\mathbf{W}' = \mathbf{W}_0 + \frac{\alpha}{r}\,\mathbf{B}\mathbf{A}$$

where $\mathbf{A} \in \mathbb{R}^{r \times d_{in}}$, $\mathbf{B} \in \mathbb{R}^{d_{out} \times r}$, rank $r \ll \min(d_{in}, d_{out})$.

**Parameter count comparison**:
- Full fine-tune: $d_{in} \times d_{out}$
- LoRA: $r(d_{in} + d_{out})$

For $d_{in} = d_{out} = 4096$, $r = 8$: LoRA uses $8 \times 8192 = 65536$ parameters vs $16{,}777{,}216$ — a $256\times$ reduction.


In [ ]:
torch.manual_seed(42)

class LoRALinear(nn.Module):
    """Linear layer with LoRA adapter. Base weights are frozen."""
    def __init__(self, d_in, d_out, rank=8, alpha=16.0):
        super().__init__()
        self.base = nn.Linear(d_in, d_out, bias=False)
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        self.scale   = alpha / rank
        # Freeze base weights
        self.base.weight.requires_grad_(False)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        return base_out + self.scale * lora_out

D_IN, D_OUT, RANK = 256, 256, 8
lora_layer = LoRALinear(D_IN, D_OUT, rank=RANK, alpha=16.0)

trainable   = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
total       = sum(p.numel() for p in lora_layer.parameters())
full_params = D_IN * D_OUT
lora_params = RANK * (D_IN + D_OUT)

print(f"Base weight frozen   : {not lora_layer.base.weight.requires_grad}")
print(f"Trainable params     : {trainable:>8}  (LoRA A + B)")
print(f"Total params         : {total:>8}")
print(f"Full fine-tune would : {full_params:>8}  trainable params")
print(f"LoRA theoretical     : {lora_params:>8}  = rank * (d_in + d_out)")
print(f"Reduction factor     : {full_params / lora_params:.1f}x")

x = torch.randn(4, D_IN)
out = lora_layer(x)
print(f"\nLoRA output shape    : {out.shape}")


## 5.9 Chinchilla Scaling Laws

Hoffmann et al. (2022) fit an empirical loss model over a wide range of model sizes $N$ and training tokens $D$:

$$\mathcal{L}(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + E$$

with best-fit constants $A = 406.4$, $B = 410.7$, $\alpha = 0.34$, $\beta = 0.28$, $E = 1.69$.

**Compute-optimal allocation** (given a FLOP budget $C \approx 6ND$):
- Optimal $N^* \propto C^{0.5}$
- Optimal $D^* \propto C^{0.5}$
- Rule of thumb: $D \approx 20N$ (train on 20 tokens per parameter)

This challenged the prior Kaplan et al. (2020) finding that favoured larger models with fewer tokens.


In [ ]:
import math

# Chinchilla constants
A, B, alpha, beta, E = 406.4, 410.7, 0.34, 0.28, 1.69

def chinchilla_loss(N, D):
    return A / N**alpha + B / D**beta + E

def optimal_N_D(C, ratio=20.0):
    """Compute-optimal N and D given FLOP budget C = 6ND."""
    # D = ratio * N  =>  C = 6 * N * ratio * N  =>  N = sqrt(C / (6*ratio))
    N_opt = math.sqrt(C / (6 * ratio))
    D_opt = ratio * N_opt
    return N_opt, D_opt

print(f"{'FLOPs (C)':>14} {'N* (params)':>14} {'D* (tokens)':>14} {'D/N ratio':>10} {'Est. Loss':>10}")
print("-" * 66)

for exp in range(21, 25):   # 1e21 to 1e24
    C = 10 ** exp
    N, D = optimal_N_D(C)
    loss = chinchilla_loss(N, D)
    print(f"1e{exp:>2}           {N:>14.3e} {D:>14.3e} {D/N:>10.1f} {loss:>10.4f}")
